In [1]:
import uuid
import logging
from base64 import b64encode
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import Any, Optional
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer

from evals.models import DeepEvalObservation, RougeObservation, ResourceObservation, GatheredResultPayload, EvalOutput
from evals import Dataset

logging.basicConfig(level=logging.INFO)
logging.getLogger(__name__).setLevel(logging.DEBUG)
logging.getLogger("httpx").setLevel(logging.WARNING)
logger = logging.getLogger(__name__)
load_dotenv()

True

## Fetch results

In [2]:
class CollectMetrics:
    def __init__(self):
        self.sentence_transformer = SentenceTransformer('all-MiniLM-L6-v2')

    # ── Helpers ───────────────────────────────────────────────────────────────

    def _extract_run_type(self, path):
        run = path.split("/")[-1].replace("llm-as-judge_", "")
        return "_".join(run.split("_")[:-1])

    def organize_runs_by_type(self, eval_res):
        runs = {}
        for path in eval_res.keys():
            ext_path = self._extract_run_type(path)
            if ext_path in runs:
                runs[ext_path].append(eval_res[path])
            else:
                runs[ext_path] = [eval_res[path]]
        return runs

    def _extract_query_metrics(self, query):
        correctness = relevancy = completeness = 0.0
        for metric in query.metrics_data:
            if metric.name == 'correctness [GEval]':
                correctness = metric.score
            elif metric.name == 'relevancy':
                relevancy = metric.score
            elif metric.name == 'completeness [GEval]':
                completeness = metric.score
        return {"correctness": correctness, "relevancy": relevancy, "completeness": completeness}

    def self_consistency_score(self, outputs: list[str]) -> float:
        """Cosine similarity between sentence embeddings of multiple outputs for the same query."""
        if len(outputs) <= 1:
            return 1.0
        embeddings = self.sentence_transformer.encode(outputs)
        sim_matrix = cosine_similarity(embeddings)
        n = len(outputs)
        mask = np.triu(np.ones((n, n), dtype=bool), k=1)
        return float(sim_matrix[mask].mean())

    def _rouge_l(self, expected_output, actual_output):
        scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
        return scorer.score(target=expected_output, prediction=actual_output)

    # ── Public collection methods ─────────────────────────────────────────────

    def collect_deepeval_metrics(self, runs) -> list[DeepEvalObservation]:
        """One DeepEvalObservation per query × eval_run from LLM-as-judge."""
        observations = []
        for run in runs:
            for res in run.results:
                for query in res.test_results:
                    m = self._extract_query_metrics(query)
                    observations.append(DeepEvalObservation(
                        dataset_name=run.dataset_name,
                        eval_run_id=run.eval_run_id,
                        query_id=query.additional_metadata.get("query_id"),
                        session_id=query.additional_metadata.get("session_id"),
                        llm_model=run.llm_model,
                        agent_type=run.agent_type,
                        name=query.name,
                        correctness=m["correctness"],
                        relevancy=m["relevancy"],
                        completeness=m["completeness"],
                        success=query.success,
                    ))
        return observations

    def collect_reference_metrics(self, runs) -> list[RougeObservation]:
        """One RougeObservation per query × eval_run for reference-based metrics."""
        observations = []
        for run in runs:
            for res in run.results:
                for query in res.test_results:
                    score = self._rouge_l(query.expected_output, query.actual_output)
                    observations.append(RougeObservation(
                        dataset_name=run.dataset_name,
                        eval_run_id=run.eval_run_id,
                        query_id=query.additional_metadata.get("query_id"),
                        session_id=query.additional_metadata.get("session_id"),
                        llm_model=run.llm_model,
                        agent_type=run.agent_type,
                        name=query.name,
                        rouge_precision=score['rougeL'].precision,
                        rouge_recall=score['rougeL'].recall,
                        rouge_fmeasure=score['rougeL'].fmeasure,
                        actual_output=query.actual_output,
                    ))
        return observations

    def collect_resource_metrics(self, runs) -> list[ResourceObservation]:
        """One ResourceObservation per query × eval_run, built from 04_results (GatheredResultPayload)."""
        observations = []
        for run in runs:
            for session in run.sessions:
                for conv in session.conversation:
                    if not conv.token_counts or not conv.time_counts:
                        continue
                    observations.append(ResourceObservation(
                        dataset_name=run.dataset_name,
                        eval_run_id=run.eval_run_id,
                        query_id=conv.query_id,
                        session_id=session.runtime_session_id,
                        llm_model=run.llm_model,
                        agent_type=run.agent_type,
                        input_tokens=conv.token_counts.input_tokens,
                        output_tokens=conv.token_counts.output_tokens,
                        total_tokens=conv.token_counts.total_tokens,
                        llm_calls=conv.token_counts.llm_calls,
                        starttime=conv.time_counts.starttime,
                        endtime=conv.time_counts.endtime,
                        duration_seconds=conv.time_counts.duration_seconds,
                    ))
        return observations

## Evaluation Analysis

In [3]:
ds = Dataset("test")
cm = CollectMetrics()
eval_res = ds.load_evaluation_results()
all_eval_runs = cm.organize_runs_by_type(eval_res)
all_eval_runs

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{}

In [6]:
coll_res = ds.load_results()
all_collected_res = cm.organize_runs_by_type(coll_res)
all_collected_res

{'google_gemini-2.5-flash_baseline': [GatheredResultPayload(dataset_name='test', project_id='b3f45644-6222-4593-8955-cf4d8e0b00d5', user_id='53d63d18-cfa1-416e-96e8-770c8f66507b', sessions=[Session(session_name='Prosjekt-initialisering', date='2020-03-15', init_query='Jeg er advokat og representerer kjøperparet Anders og Berit Kristiansen i en eiendomskjøpssak. De kjøpte en eiendom på Fjellveien 42A i Stavanger kommune den 1. juni 2019, med overtakelse 1. august 2019. \nVi er nå i mars 2020 og det har dukket opp flere problemer med eiendommen.  Jeg trenger din hjelp til å organisere saksinnholdet, identifisere de juridiske problemstillingene, og vurdere mulige tiltak.', conversation=[ConversationTurn(input='Gi meg en kort og konsis oppsummering av sakens faktiske bakgrunn og utvikling så langt, basert på dokumentene jeg har lastet opp. \nFokuser på de viktigste hendelsene og problemstillingene. Hva er kjernen i saken?', answer="Anders og Berit Kristiansen kjøpte en eiendom på Fjellveie

In [7]:
deepeval_obs = []
for runs in all_eval_runs.values():
    deepeval_obs.extend(cm.collect_deepeval_metrics(runs))

deepeval_obs[:2], len(deepeval_obs)

([], 0)

In [8]:
rouge_obs = []
for runs in all_eval_runs.values():
    rouge_obs.extend(cm.collect_reference_metrics(runs))

rouge_obs[:2], len(rouge_obs)

([], 0)

In [9]:
gathered_res = ds.load_results()
resource_obs = cm.collect_resource_metrics(list(gathered_res.values()))

resource_obs[:2], len(resource_obs)

([ResourceObservation(starttime=datetime.datetime(2026, 3, 2, 14, 26, 13, 777326), endtime=datetime.datetime(2026, 3, 2, 14, 26, 25, 821724), duration_seconds=12.044398, input_tokens=0, output_tokens=0, total_tokens=0, llm_calls=0, dataset_name='test', eval_run_id='3c60bc74-d9a5-4053-aa0c-3f346dd884fc', query_id='f3f226bb-eb5c-4b87-8d96-dfa425614f42', session_id='8fb46d97-d328-40d4-becf-21fb8181c1c9', llm_model='google_gemini-2.5-flash', agent_type='baseline'),
  ResourceObservation(starttime=datetime.datetime(2026, 3, 2, 14, 26, 25, 821923), endtime=datetime.datetime(2026, 3, 2, 14, 26, 43, 952835), duration_seconds=18.130912, input_tokens=0, output_tokens=0, total_tokens=0, llm_calls=0, dataset_name='test', eval_run_id='3c60bc74-d9a5-4053-aa0c-3f346dd884fc', query_id='070f570b-f9d0-454f-90ca-84f66ede1a31', session_id='8fb46d97-d328-40d4-becf-21fb8181c1c9', llm_model='google_gemini-2.5-flash', agent_type='baseline')],
 15)

## Statistical Modeling

In [10]:
import numpy as np
from scipy import stats 
import statsmodels.api as sm
from statsmodels.stats.power import tt_ind_solve_power  

In [11]:
class StatisticalMetrics:
    def __init__(self, significance_level: float = 0.05):
        self.significance_level = significance_level

    def check_normality(self, observations: list[DeepEvalObservation], metric: str = "correctness"):
        for agent in ["baseline", "baseline_rag", "custom"]:
            values = [getattr(obs, metric) for obs in observations if obs.agent_type == agent]
            if not values:
                continue
            result = stats.shapiro(values)
            status = "✅" if result.pvalue > self.significance_level else "❌"
            print(f"{status} {agent}: {result}")

    def t_test(self, custom_eval: list[float] | np.ndarray, comparison_eval: list[float] | np.ndarray):
        t_stat, p = stats.ttest_ind(custom_eval, comparison_eval, equal_var=False, alternative='greater')
        print(f"Custom vs. Comparison: t={t_stat:.2f}, p={p:.4f}")
        if p < self.significance_level:
            print("Custom slår Comparison (p<0.05).")

## One-sided Hypothesis Tests

**For quality/performance metrics** (correctness, relevancy, completeness, passrate):

$$
H_0: \mu_{\text{custom}} \leq \mu_{\text{baseline}} \qquad H_1: \mu_{\text{custom}} > \mu_{\text{baseline}}
$$

(and analogously vs. baseline+RAG)

**For cost/efficiency metrics** (token usage, inference time):

$$
H_0: \mu_{\text{custom}} \geq \mu_{\text{baseline}} \qquad H_1: \mu_{\text{custom}} < \mu_{\text{baseline}}
$$

(and analogously vs. baseline+RAG)

**Notation:**

- $\mu_{\text{custom}}$ = population mean for the custom agent  
- $\mu_{\text{baseline}}$ = population mean for the baseline agent  
- $\mu_{\text{baseline+RAG}}$ = population mean for baseline with RAG  

All tests are one-tailed / one-sided.

In [12]:
from collections import Counter
Counter(obs.agent_type for obs in deepeval_obs)

Counter()

## EDA
### Exploratory data analysis

In [13]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd

In [14]:
df_deepeval  = pd.DataFrame(obs.model_dump() for obs in deepeval_obs)
df_rouge     = pd.DataFrame(obs.model_dump() for obs in rouge_obs)
df_resource  = pd.DataFrame(obs.model_dump() for obs in resource_obs)

# self-consistency: computed per query across eval_runs
sc = df_rouge.groupby(["query_id", "agent_type"])["actual_output"].apply(
    lambda outputs: cm.self_consistency_score(outputs.tolist())
).reset_index(name="self_consistency")
df_rouge = df_rouge.merge(sc, on=["query_id", "agent_type"], how="left")

df_deepeval.info()

KeyError: 'query_id'

In [ ]:
px.histogram(data_frame=df_deepeval, x="correctness", color="agent_type", nbins=100, title="Correctness Distribution by Agent Type")

In [15]:
px.histogram(data_frame=df_resource, x="total_tokens", color="agent_type", nbins=20, title="Total Tokens Used by Agent Type")

In [16]:
df_resource

,starttime,endtime,duration_seconds,input_tokens,output_tokens,total_tokens,llm_calls,dataset_name,eval_run_id,query_id,session_id,llm_model,agent_type
0,2026-03-02 14:26:13.777326,2026-03-02 14:26:25.821724,12.044398,0,0,0,0,test,3c60bc74-d9a5-4053-aa0c-3f346dd884fc,f3f226bb-eb5c-4b87-8d96-dfa425614f42,8fb46d97-d328-40d4-becf-21fb8181c1c9,google_gemini-2.5-flash,baseline
1,2026-03-02 14:26:25.821923,2026-03-02 14:26:43.952835,18.130912,0,0,0,0,test,3c60bc74-d9a5-4053-aa0c-3f346dd884fc,070f570b-f9d0-454f-90ca-84f66ede1a31,8fb46d97-d328-40d4-becf-21fb8181c1c9,google_gemini-2.5-flash,baseline
2,2026-03-02 14:26:43.952894,2026-03-02 14:27:04.890012,20.937118,0,0,0,0,test,3c60bc74-d9a5-4053-aa0c-3f346dd884fc,335b247f-0a2b-4eef-85f8-5cc0e2c6a2e2,8fb46d97-d328-40d4-becf-21fb8181c1c9,google_gemini-2.5-flash,baseline
3,2026-03-02 14:27:04.890060,2026-03-02 14:27:16.779737,11.889677,0,0,0,0,test,3c60bc74-d9a5-4053-aa0c-3f346dd884fc,b309a1f0-2399-461a-9739-047bf840516b,8fb46d97-d328-40d4-becf-21fb8181c1c9,google_gemini-2.5-flash,baseline
4,2026-03-02 14:27:16.779816,2026-03-02 14:27:19.987000,3.207184,0,0,0,0,test,3c60bc74-d9a5-4053-aa0c-3f346dd884fc,82d6385b-3f42-4c89-8df6-82cea426f45a,8fb46d97-d328-40d4-becf-21fb8181c1c9,google_gemini-2.5-flash,baseline
5,2026-03-02 14:26:22.666678,2026-03-02 14:26:36.854925,14.188247,0,0,0,0,test,da24de98-67b4-44d9-9494-0775d6222254,f3f226bb-eb5c-4b87-8d96-dfa425614f42,5d6007bc-b38a-4cd0-9add-2f13c0418e07,google_gemini-2.5-flash,baseline_rag
6,2026-03-02 14:26:36.854990,2026-03-02 14:26:56.181407,19.326417,0,0,0,0,test,da24de98-67b4-44d9-9494-0775d6222254,070f570b-f9d0-454f-90ca-84f66ede1a31,5d6007bc-b38a-4cd0-9add-2f13c0418e07,google_gemini-2.5-flash,baseline_rag
7,2026-03-02 14:26:56.181468,2026-03-02 14:27:12.840971,16.659503,0,0,0,0,test,da24de98-67b4-44d9-9494-0775d6222254,335b247f-0a2b-4eef-85f8-5cc0e2c6a2e2,5d6007bc-b38a-4cd0-9add-2f13c0418e07,google_gemini-2.5-flash,baseline_rag
8,2026-03-02 14:27:12.841044,2026-03-02 14:27:28.013410,15.172366,0,0,0,0,test,da24de98-67b4-44d9-9494-0775d6222254,b309a1f0-2399-461a-9739-047bf840516b,5d6007bc-b38a-4cd0-9add-2f13c0418e07,google_gemini-2.5-flash,baseline_rag
9,2026-03-02 14:27:28.013510,2026-03-02 14:27:31.625518,3.612008,0,0,0,0,test,da24de98-67b4-44d9-9494-0775d6222254,82d6385b-3f42-4c89-8df6-82cea426f45a,5d6007bc-b38a-4cd0-9add-2f13c0418e07,google_gemini-2.5-flash,baseline_rag


In [16]:
from langsmith import Client as LangSmithClient
import os
from dotenv import load_dotenv

def get_session_token_counts(runtime_session_id: str) -> dict:
    """Query LangSmith for per-query token usage for a single session via thread_id."""
    client = LangSmithClient()
    project_name = os.getenv("LANGCHAIN_PROJECT", "default")
    runs = list(client.list_runs(
        project_name=project_name,
        filter=f'has(metadata, \'{{"thread_id": "{runtime_session_id}"}}\')',
        run_type="llm",
    ))
    per_query: dict[str, dict] = {}
    for r in runs:
        qid = (r.extra or {}).get("metadata", {}).get("query_id", "unknown")
        entry = per_query.setdefault(qid, {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0, "llm_calls": 0})
        entry["input_tokens"] += r.prompt_tokens or 0
        entry["output_tokens"] += r.completion_tokens or 0
        entry["total_tokens"] += r.total_tokens or 0
        entry["llm_calls"] += 1
    return {
        "runtime_session_id": runtime_session_id,
        "input_tokens": sum(r.prompt_tokens or 0 for r in runs),
        "output_tokens": sum(r.completion_tokens or 0 for r in runs),
        "total_tokens": sum(r.total_tokens or 0 for r in runs),
        "llm_calls": len(runs),
        "per_query": per_query,
    }


In [14]:
ds = Dataset("test")
res = ds.load_results()

In [23]:
blobs = ds.bucket.list_blobs(prefix="datasets/test/01_data/")

In [25]:
all = []
for blob in blobs:
    content = blob.download_as_text()
    all.append(f'FILE : {blob.name}\n{content}\n---\n')

In [27]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

In [28]:
len(enc.encode("\n".join(all)))

9625

In [7]:
ds = Dataset("test")

NameError: name 'Dataset' is not defined

In [5]:
from langsmith import Client as LangSmithClient
import os
from dotenv import load_dotenv
load_dotenv()

runtime_session_id = "d34fe6b0-3446-4e91-bdc1-669f83c8e5e5"
client = LangSmithClient()
project_name = os.getenv("LANGCHAIN_PROJECT", "default")
runs = list(client.list_runs(
    project_name=project_name,
    filter=f'has(metadata, \'{{"thread_id": "{runtime_session_id}"}}\')',
    run_type="llm",
))

In [2]:
from evals.dataset_module import Dataset
ds = Dataset("test")

In [3]:
res = ds.load_results()
res

{'datasets/test/04_results/google_gemini-2.5-flash_baseline_bb836461-10bd-46f1-8591-8d7f728debde.json': GatheredResultPayload(dataset_name='test', project_id='b3f45644-6222-4593-8955-cf4d8e0b00d5', user_id='53d63d18-cfa1-416e-96e8-770c8f66507b', sessions=[Session(session_name='Prosjekt-initialisering', date='2020-03-15', init_query='Jeg er advokat og representerer kjøperparet Anders og Berit Kristiansen i en eiendomskjøpssak. De kjøpte en eiendom på Fjellveien 42A i Stavanger kommune den 1. juni 2019, med overtakelse 1. august 2019. \nVi er nå i mars 2020 og det har dukket opp flere problemer med eiendommen.  Jeg trenger din hjelp til å organisere saksinnholdet, identifisere de juridiske problemstillingene, og vurdere mulige tiltak.', init_query_id='7824fa60-b42a-4977-8690-ef15f1c37ea3', init_query_token_count=TokenCount(input_tokens=0, output_tokens=0, total_tokens=0, llm_calls=0), init_query_time_count=None, conversation=[ConversationTurn(input='Gi meg en kort og konsis oppsummering 

In [4]:
for r in res.values():
    ds.update_token_counts(r)

INFO:evals.dataset_module:📊 Tokens — in: 87117  out: 23996  total: 111113  calls: 6
INFO:evals.dataset_module:Results saved to datasets/test/04_results/google_gemini-2.5-flash_baseline_bb836461-10bd-46f1-8591-8d7f728debde.json
INFO:evals.dataset_module:📊 Tokens — in: 106447  out: 20462  total: 126909  calls: 8
INFO:evals.dataset_module:Results saved to datasets/test/04_results/google_gemini-2.5-flash_baseline_rag_43ce09fd-3e67-4120-93a7-64a48c95c33f.json
INFO:evals.dataset_module:📊 Tokens — in: 103418  out: 42370  total: 145788  calls: 11
INFO:evals.dataset_module:Results saved to datasets/test/04_results/google_gemini-2.5-flash_custom_7578ca4f-2a3d-4400-bf96-305bab41bdd6.json


In [9]:
r.sessions[0].token_counts

TokenCount(input_tokens=114039, output_tokens=31806, total_tokens=145845, llm_calls=14)

In [6]:
ds.update_token_counts(r)

INFO:evals.dataset_module:📊 Tokens — in: 114039  out: 31806  total: 145845  calls: 14
INFO:evals.dataset_module:Results saved to datasets/test/04_results/google_gemini-2.5-flash_custom_7be78b05-24d4-4a10-b563-a73a8792c445.json


In [16]:
from evals.langsmith_module import get_session_token_counts
raw_tokens = get_session_token_counts(r.sessions[0].runtime_session_id)
raw_tokens

{'runtime_session_id': '188ff836-215b-41ce-9fe2-64340cb5e554',
 'input_tokens': 114039,
 'output_tokens': 31806,
 'total_tokens': 145845,
 'llm_calls': 14,
 'per_query': {'82d6385b-3f42-4c89-8df6-82cea426f45a': {'input_tokens': 27757,
   'output_tokens': 1600,
   'total_tokens': 29357,
   'llm_calls': 2},
  'b309a1f0-2399-461a-9739-047bf840516b': {'input_tokens': 23973,
   'output_tokens': 2561,
   'total_tokens': 26534,
   'llm_calls': 2},
  '335b247f-0a2b-4eef-85f8-5cc0e2c6a2e2': {'input_tokens': 16154,
   'output_tokens': 991,
   'total_tokens': 17145,
   'llm_calls': 2},
  '070f570b-f9d0-454f-90ca-84f66ede1a31': {'input_tokens': 20690,
   'output_tokens': 3729,
   'total_tokens': 24419,
   'llm_calls': 4},
  'f3f226bb-eb5c-4b87-8d96-dfa425614f42': {'input_tokens': 8869,
   'output_tokens': 1250,
   'total_tokens': 10119,
   'llm_calls': 1},
  '7824fa60-b42a-4977-8690-ef15f1c37ea3': {'input_tokens': 16596,
   'output_tokens': 21675,
   'total_tokens': 38271,
   'llm_calls': 3}}}